# Chicago Taxi Trips — week of 11–17 September 2023

A descriptive summary of one week of reported taxi trips: when people ride, where trips start, how they pay and tip, and which companies carry the trips.

**Summary placeholder — replaced at the end of the notebook build.**

---
## 1. Setup and data loading
**Conventions used throughout**
- Every figure is *reported trips*, not total market volume (the City notes not all trips are reported).
- "Revenue" means `trip_total` as reported. It includes tips, tolls and extras, so it is what the rider was charged, not what the company or driver kept.
- Timestamps are local time, rounded to 15 minutes. The brief and `dataset.md` don't state a time zone, so I assume Chicago local time. Durations use `trip_seconds`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

FIG_DIR = Path("figures/analysis")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Chart style: white surface, thin marks, recessive grid, no dual axes.
BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#9a9a96", "#0b0b0b"
plt.rcParams.update({
    "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#52514e", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "font.size": 10,
})

In [2]:
raw = pd.read_csv("trips.csv", dtype={"trip_id": "string", "taxi_id": "string"})
raw["start"] = pd.to_datetime(raw["trip_start_timestamp"])
raw["end"] = pd.to_datetime(raw["trip_end_timestamp"])

print("Rows, columns:", raw.shape, "(brief says 135,768 rows)")
print("Start range  :", raw["start"].min(), "->", raw["start"].max())
print("Duplicate trip_id:", raw["trip_id"].duplicated().sum(), "| fully duplicated rows:", raw.drop(columns=["start", "end"]).duplicated().sum())
print("Trips per calendar date:")
print(raw["start"].dt.date.value_counts().sort_index().to_string())

Rows, columns: (135768, 25) (brief says 135,768 rows)
Start range  : 2023-09-11 00:00:00 -> 2023-09-17 23:45:00
Duplicate trip_id: 0 | fully duplicated rows: 0
Trips per calendar date:
start
2023-09-11    22190
2023-09-12    22040
2023-09-13    22694
2023-09-14    22531
2023-09-15    20226
2023-09-16    12719
2023-09-17    13368


In [3]:
miss = raw.drop(columns=["start", "end"]).isna().sum()
miss = pd.DataFrame({"missing": miss, "pct": miss / len(raw) * 100})
print(miss[miss.missing > 0].to_string())

                            missing   pct
trip_end_timestamp                1  0.00
trip_seconds                     33  0.02
pickup_census_tract           71404 52.59
dropoff_census_tract          73842 54.39
pickup_community_area          3815  2.81
dropoff_community_area        13794 10.16
fare                            324  0.24
tips                            324  0.24
tolls                           324  0.24
extras                          324  0.24
trip_total                      324  0.24
pickup_centroid_latitude       3775  2.78
pickup_centroid_longitude      3775  2.78
pickup_centroid_location       3775  2.78
dropoff_centroid_latitude     12872  9.48
dropoff_centroid_longitude    12872  9.48
dropoff_centroid_location     12872  9.48
